1. Create DataFrame

In [0]:
data = [
(101,"Arjun Reddy","Hyderabad","Cardiology",5000),
(102,"Sneha Kapoor","Delhi","Orthopedics",3000),
(103,"Rahul Sharma","Mumbai","Dermatology",1500),
(104,"Priya Nair","Bangalore","Cardiology",5000),
(105,"Vikram Singh","Chennai","Neurology",7000)
]
columns = ["visit_id","patient_name","city","department","consultation_fee"]
df = spark.createDataFrame(data, columns)
display(df)

visit_id,patient_name,city,department,consultation_fee
101,Arjun Reddy,Hyderabad,Cardiology,5000
102,Sneha Kapoor,Delhi,Orthopedics,3000
103,Rahul Sharma,Mumbai,Dermatology,1500
104,Priya Nair,Bangalore,Cardiology,5000
105,Vikram Singh,Chennai,Neurology,7000


2. Write Data as Parquet


In [0]:
df.write \
 .mode("overwrite") \
 .parquet("/tmp/patient_parquet")


3. Read Parquet Data

In [0]:
parquet_df = spark.read.parquet("/tmp/patient_parquet")
display(parquet_df)


visit_id,patient_name,city,department,consultation_fee
101,Arjun Reddy,Hyderabad,Cardiology,5000
104,Priya Nair,Bangalore,Cardiology,5000
103,Rahul Sharma,Mumbai,Dermatology,1500
105,Vikram Singh,Chennai,Neurology,7000
102,Sneha Kapoor,Delhi,Orthopedics,3000


4. Schema Inspection

In [0]:
parquet_df.printSchema()

root
 |-- visit_id: long (nullable = true)
 |-- patient_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- department: string (nullable = true)
 |-- consultation_fee: long (nullable = true)



Column Projection (Read Specific
Columns)


In [0]:
spark.read.parquet("/tmp/patient_parquet") \
 .select("patient_name","city") \
 .show()


+------------+---------+
|patient_name|     city|
+------------+---------+
| Arjun Reddy|Hyderabad|
|  Priya Nair|Bangalore|
|Rahul Sharma|   Mumbai|
|Vikram Singh|  Chennai|
|Sneha Kapoor|    Delhi|
+------------+---------+



6. Filtering Data

In [0]:
spark.read.parquet("/tmp/patient_parquet") \
 .filter("consultation_fee > 3000") \
 .show()

+--------+------------+---------+----------+----------------+
|visit_id|patient_name|     city|department|consultation_fee|
+--------+------------+---------+----------+----------------+
|     101| Arjun Reddy|Hyderabad|Cardiology|            5000|
|     104|  Priya Nair|Bangalore|Cardiology|            5000|
|     105|Vikram Singh|  Chennai| Neurology|            7000|
+--------+------------+---------+----------+----------------+



7. Partitioned Parquet Write

In [0]:
df.write \
 .mode("overwrite") \
 .partitionBy("city") \
 .parquet("/tmp/patient_parquet_partitioned")


8. Read Partitioned Data

In [0]:
spark.read.parquet("/tmp/patient_parquet_partitioned").show()

+--------+------------+-----------+----------------+---------+
|visit_id|patient_name| department|consultation_fee|     city|
+--------+------------+-----------+----------------+---------+
|     103|Rahul Sharma|Dermatology|            1500|   Mumbai|
|     102|Sneha Kapoor|Orthopedics|            3000|    Delhi|
|     101| Arjun Reddy| Cardiology|            5000|Hyderabad|
|     105|Vikram Singh|  Neurology|            7000|  Chennai|
|     104|  Priya Nair| Cardiology|            5000|Bangalore|
+--------+------------+-----------+----------------+---------+



9. Partition Pruning (Performance
Concept)

In [0]:
spark.read.parquet("/tmp/patient_parquet_partitioned") \
 .filter("city = 'Hyderabad'") \
 .show

<bound method DataFrame.show of DataFrame[visit_id: bigint, patient_name: string, department: string, consultation_fee: bigint, city: string]>

10. Append Mode

In [0]:
new_data = [
(106,"Ananya Das","Kolkata","Orthopedics",3000)
]
new_df = spark.createDataFrame(new_data, columns)
new_df.write \
 .mode("append") \
 .parquet("/tmp/patient_parquet")

11.  Overwrite Mode

In [0]:
df.write \
 .mode("overwrite") \
 .parquet("/tmp/patient_parquet")

Exercises
1. Write DataFrame to Parquet
2. Read and display
3. Filter high-value records
4. Write partitioned Parquet
5. Read only one partition
6. Append new data


In [0]:
df.write.mode("overwrite").parquet("/tmp/patient_parquet")

In [0]:
parquet_df = spark.read.parquet("/tmp/patient_parquet")
display(parquet_df)

visit_id,patient_name,city,department,consultation_fee
101,Arjun Reddy,Hyderabad,Cardiology,5000
104,Priya Nair,Bangalore,Cardiology,5000
103,Rahul Sharma,Mumbai,Dermatology,1500
105,Vikram Singh,Chennai,Neurology,7000
102,Sneha Kapoor,Delhi,Orthopedics,3000


In [0]:
spark.read.parquet("/tmp/patient_parquet") \
.filter("consultation_fee > 3000") \
.show()

+--------+------------+---------+----------+----------------+
|visit_id|patient_name|     city|department|consultation_fee|
+--------+------------+---------+----------+----------------+
|     101| Arjun Reddy|Hyderabad|Cardiology|            5000|
|     104|  Priya Nair|Bangalore|Cardiology|            5000|
|     105|Vikram Singh|  Chennai| Neurology|            7000|
+--------+------------+---------+----------+----------------+



In [0]:
df.write.mode("overwrite") \
.partitionBy("city") \
.parquet("/tmp/patient_parquet_partitioned")

In [0]:
spark.read.parquet("/tmp/patient_parquet_partitioned") \
.filter("city = 'Hyderabad'") \
.show()

+--------+------------+----------+----------------+---------+
|visit_id|patient_name|department|consultation_fee|     city|
+--------+------------+----------+----------------+---------+
|     101| Arjun Reddy|Cardiology|            5000|Hyderabad|
+--------+------------+----------+----------------+---------+



In [0]:
new_data = [(106,"Ananya Das","Kolkata","Orthopedics",3000)]
new_df = spark.createDataFrame(new_data, df.columns)

new_df.write.mode("append").parquet("/tmp/patient_parquet")

10. Explain difference between Parquet and Delta

 **Parquet**

Columnar storage format

Used for fast read performance

No support for UPDATE / DELETE

No ACID transactions

No versioning (no history)

Schema enforcement is limited

**Delta**

Built on top of Parquet

Supports UPDATE, DELETE, MERGE

Provides ACID transactions (data reliability)

Supports Time Travel (version history)

Has schema enforcement & evolution

Better for data pipelines and production use